In [3]:
from pathlib import Path

BASE_DIR = Path(".")

INPUT_DIR = BASE_DIR / "input"
OUTPUT_DIR = BASE_DIR / "output"
REPORTS_DIR = BASE_DIR / "reports"

# Create required directories
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project directories ready.")

Project directories ready.


Before performing any video analysis, the system checks whether the FFmpeg tools are correctly installed. The program searches for the ffmpeg and ffprobe executables and prints their locations if they are available.

In [4]:
# test
import shutil, subprocess

print("ffmpeg:", shutil.which("ffmpeg"))
print("ffprobe:", shutil.which("ffprobe"))

subprocess.run(["ffprobe", "-version"], check=True)
subprocess.run(["ffmpeg", "-version"], check=True)


ffmpeg: C:\Users\me\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.1.1-full_build\bin\ffmpeg.EXE
ffprobe: C:\Users\me\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.1.1-full_build\bin\ffprobe.EXE


CompletedProcess(args=['ffmpeg', '-version'], returncode=0)

The next step is to locate all video files within the input directory. The program scans the directory recursively and selects files that match common video extensions such as MP4, MKV, MOV, AVI, and WebM.

In [5]:
VIDEO_EXTS = {".mp4", ".mkv", ".mov", ".avi", ".webm", ".m4v"}

video_files = sorted([p for p in INPUT_DIR.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTS])
if not video_files:
    raise FileNotFoundError(
        "No input videos found. Add video files to the input/ folder before running."
    )

print(f"Found {len(video_files)} video file(s) in {INPUT_DIR}:")
for p in video_files[:50]:
    print(" -", p.name)
if len(video_files) > 50:
    print(" ...")


Found 5 video file(s) in input:
 - Cosmos_War_of_the_Planets.mp4
 - Last_man_on_earth_1964.mov
 - The_Gun_and_the_Pulpit.avi
 - The_Hill_Gang_Rides_Again.mp4
 - Voyage_to_the_Planet_of_Prehistoric_Women.mp4


To analyse the technical properties of each video, the program uses the ffprobe tool from FFmpeg. This utility can inspect media files and return detailed metadata describing their format and stream information.

In [6]:
import json, subprocess
from pathlib import Path

def ffprobe_json(path: Path) -> dict:
    cmd = [
        "ffprobe",
        "-v", "error",
        "-print_format", "json",
        "-show_format",
        "-show_streams",
        str(path)
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"ffprobe failed for {path.name}:\n{proc.stderr}")
    return json.loads(proc.stdout)

# quick smoke test on first file
probe0 = ffprobe_json(video_files[0])
print("OK probed:", video_files[0].name)
print("format_name:", probe0.get("format", {}).get("format_name"))
print("num_streams:", len(probe0.get("streams", [])))


OK probed: Cosmos_War_of_the_Planets.mp4
format_name: mov,mp4,m4a,3gp,3g2,mj2
num_streams: 2


The metadata returned by ffprobe contains many fields, so the next step is to extract the specific values needed for validation. The program retrieves properties such as container format, video codec, audio codec, resolution, frame rate, bitrate, and number of channels.

In [7]:
def parse_fraction(frac: str) -> float | None:
    if not frac:
        return None
    if isinstance(frac, (int, float)):
        return float(frac)
    if "/" in frac:
        n, d = frac.split("/", 1)
        try:
            n, d = float(n), float(d)
            return n / d if d != 0 else None
        except:
            return None
    try:
        return float(frac)
    except:
        return None

def bps_to_mbps(bps) -> float | None:
    try:
        return int(bps) / 1_000_000
    except:
        return None

def bps_to_kbps(bps) -> float | None:
    try:
        return int(bps) / 1_000
    except:
        return None

def is_16_9(width: int | None, height: int | None, tol: float = 0.02) -> bool:
    if not width or not height or height == 0:
        return False
    return abs((width/height) - (16/9)) <= tol*(16/9)


In [8]:
def extract_meta(probe: dict) -> dict:
    fmt = probe.get("format", {}) or {}
    streams = probe.get("streams", []) or []

    format_name_raw = (fmt.get("format_name") or "").lower()
    container = "mp4" if "mp4" in format_name_raw else (format_name_raw.split(",")[0] if format_name_raw else None)

    v = next((s for s in streams if s.get("codec_type") == "video"), None)
    a = next((s for s in streams if s.get("codec_type") == "audio"), None)

    vcodec = (v.get("codec_name") if v else None)
    acodec = (a.get("codec_name") if a else None)

    width = int(v.get("width")) if v and v.get("width") else None
    height = int(v.get("height")) if v and v.get("height") else None

    fps = None
    if v:
        fps = parse_fraction(v.get("avg_frame_rate")) or parse_fraction(v.get("r_frame_rate"))

    dar = v.get("display_aspect_ratio") if v else None
    aspect_ok = (dar == "16:9") if isinstance(dar, str) else is_16_9(width, height)

    v_mbps = bps_to_mbps(v.get("bit_rate")) if v else None
    a_kbps = bps_to_kbps(a.get("bit_rate")) if a else None
    channels = int(a.get("channels")) if a and a.get("channels") else None

    return {
        "container": container,
        "format_name_raw": format_name_raw,

        "vcodec": vcodec.lower() if isinstance(vcodec, str) else None,
        "acodec": acodec.lower() if isinstance(acodec, str) else None,

        "width": width,
        "height": height,
        "fps": fps,

        "dar": dar,
        "aspect_ok": aspect_ok,

        "v_bitrate_mbps": v_mbps,
        "a_bitrate_kbps": a_kbps,
        "channels": channels,
    }

# test on first file
meta0 = extract_meta(probe0)
meta0


{'container': 'mp4',
 'format_name_raw': 'mov,mp4,m4a,3gp,3g2,mj2',
 'vcodec': 'h264',
 'acodec': 'aac',
 'width': 628,
 'height': 354,
 'fps': 29.97002997002997,
 'dar': '314:177',
 'aspect_ok': False,
 'v_bitrate_mbps': 2.989377,
 'a_bitrate_kbps': 317.103,
 'channels': 2}

Once the relevant metadata fields are extracted, each video file is evaluated against the previously defined specification. The validation function checks whether every property falls within the required constraints.

In [9]:
SPEC = {
    "container": "mp4",
    "vcodec": {"hevc"},      # h.265 typically shows as "hevc"
    "acodec": {"aac"},
    "fps": 25.0,
    "fps_tol": 0.2,
    "width": 640,
    "height": 360,
    "v_mbps_min": 2.0,
    "v_mbps_max": 5.0,
    "a_kbps_max": 256.0,
    "channels": 2
}

def validate(meta: dict) -> tuple[bool, list[str]]:
    probs = []

    if meta.get("container") != SPEC["container"]:
        probs.append("container")
    if meta.get("vcodec") not in SPEC["vcodec"]:
        probs.append("video_codec")
    if meta.get("acodec") not in SPEC["acodec"]:
        probs.append("audio_codec")

    fps = meta.get("fps")
    if fps is None or abs(fps - SPEC["fps"]) > SPEC["fps_tol"]:
        probs.append("frame_rate")

    if not meta.get("aspect_ok"):
        probs.append("aspect_ratio")

    if meta.get("width") != SPEC["width"] or meta.get("height") != SPEC["height"]:
        probs.append("resolution")

    v = meta.get("v_bitrate_mbps")
    if v is None or not (SPEC["v_mbps_min"] <= v <= SPEC["v_mbps_max"]):
        probs.append("video_bit_rate")

    a = meta.get("a_bitrate_kbps")
    if a is None or a > SPEC["a_kbps_max"]:
        probs.append("audio_bit_rate")

    if meta.get("channels") != SPEC["channels"]:
        probs.append("audio_channels")

    return (len(probs) == 0), probs


In [10]:
results = []
for p in video_files:
    probe = ffprobe_json(p)
    meta = extract_meta(probe)
    ok, probs = validate(meta)
    results.append({"path": p, "ok": ok, "problems": probs, "meta": meta})

ok_count = sum(1 for r in results if r["ok"])
print(f"OK: {ok_count} / {len(results)}")

for r in results:
    if not r["ok"]:
        print("NOT OK:", r["path"].name, "->", r["problems"])


OK: 0 / 5
NOT OK: Cosmos_War_of_the_Planets.mp4 -> ['video_codec', 'frame_rate', 'aspect_ratio', 'resolution', 'audio_bit_rate']
NOT OK: Last_man_on_earth_1964.mov -> ['video_codec', 'audio_codec', 'frame_rate', 'video_bit_rate', 'audio_bit_rate']
NOT OK: The_Gun_and_the_Pulpit.avi -> ['container', 'video_codec', 'audio_codec', 'resolution', 'video_bit_rate', 'audio_bit_rate']
NOT OK: The_Hill_Gang_Rides_Again.mp4 -> ['video_codec', 'video_bit_rate']
NOT OK: Voyage_to_the_Planet_of_Prehistoric_Women.mp4 -> ['audio_codec', 'frame_rate', 'video_bit_rate', 'audio_bit_rate']


After validating all video files, the results are summarised in a text report. The report lists each video file and indicates whether it satisfies the required format specification.

In [11]:
from datetime import datetime

report_path = REPORTS_DIR / "format_report.txt"
lines = []
lines.append("Narbonne Online Film Festival - Format Compliance Report")
lines.append(f"Generated: {datetime.now().isoformat(timespec='seconds')}")
lines.append("")
lines.append("Target spec:")
lines.append("- container: mp4")
lines.append("- video codec: h.265 (ffprobe: hevc)")
lines.append("- audio codec: aac")
lines.append("- frame rate: 25 FPS")
lines.append("- aspect ratio: 16:9")
lines.append("- resolution: 640 x 360")
lines.append("- video bit rate: 2–5 Mb/s")
lines.append("- audio bit rate: up to 256 kb/s")
lines.append("- audio channels: stereo (2 ch)")
lines.append("")
lines.append("Results:")
lines.append("")

for r in results:
    name = r["path"].name
    if r["ok"]:
        lines.append(f"{name} -> OK")
    else:
        m = r["meta"]
        probs = ", ".join(r["problems"])
        observed = (
            f"container={m.get('container')} | vcodec={m.get('vcodec')} | acodec={m.get('acodec')} | "
            f"fps={m.get('fps')} | res={m.get('width')}x{m.get('height')} | 16:9={m.get('aspect_ok')} | "
            f"v_mbps={m.get('v_bitrate_mbps')} | a_kbps={m.get('a_bitrate_kbps')} | ch={m.get('channels')}"
        )
        lines.append(f"{name} -> NOT OK | problematic_fields: {probs}")
        lines.append(f"  observed: {observed}")

report_path.write_text("\n".join(lines), encoding="utf-8")
print("Wrote:", report_path.resolve())


Wrote: C:\Users\me\Desktop\Portfolio\Projects\Intelligent Signal Processing\isp proj\exercise-3-media-processing\reports\format_report.txt


If a video does not meet the required format specification, the program can automatically convert it into a compliant version. The conversion process uses FFmpeg to re-encode the video using the required codec and format settings.

In [12]:
def out_path_for(src: Path) -> Path:
    return OUTPUT_DIR / f"{src.stem}_formatOK.mp4"

def convert_to_spec(src: Path, dst: Path):
    cmd = [
        "ffmpeg", "-y",
        "-i", str(src),

        # video: h.265 + 25fps + 640x360 (pad to keep 16:9 if needed)
        "-c:v", "libx265",
        "-r", "25",
        "-vf", "scale=640:360:force_original_aspect_ratio=decrease,pad=640:360:(ow-iw)/2:(oh-ih)/2",

        # bitrate within 2–5 Mb/s (choose a safe mid target)
        "-b:v", "3M",
        "-maxrate", "5M",
        "-bufsize", "10M",

        # audio: aac stereo <=256k
        "-c:a", "aac",
        "-b:a", "256k",
        "-ac", "2",

        "-movflags", "+faststart",
        str(dst)
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr)

to_convert = [r for r in results if not r["ok"]]
print("To convert:", len(to_convert))

for r in to_convert:
    src = r["path"]
    dst = out_path_for(src)
    convert_to_spec(src, dst)
    print("Converted:", src.name, "->", dst.name)


To convert: 5
Converted: Cosmos_War_of_the_Planets.mp4 -> Cosmos_War_of_the_Planets_formatOK.mp4
Converted: Last_man_on_earth_1964.mov -> Last_man_on_earth_1964_formatOK.mp4
Converted: The_Gun_and_the_Pulpit.avi -> The_Gun_and_the_Pulpit_formatOK.mp4
Converted: The_Hill_Gang_Rides_Again.mp4 -> The_Hill_Gang_Rides_Again_formatOK.mp4
Converted: Voyage_to_the_Planet_of_Prehistoric_Women.mp4 -> Voyage_to_the_Planet_of_Prehistoric_Women_formatOK.mp4


In [28]:
converted = sorted([p for p in OUTPUT_DIR.glob("*_formatOK.mp4") if p.is_file()])
print("Converted files:", len(converted))

for p in converted:
    meta = extract_meta(ffprobe_json(p))
    ok, probs = validate(meta)
    print(p.name, "->", "OK" if ok else f"FAIL {probs}")


Converted files: 5
Cosmos_War_of_the_Planets_formatOK.mp4 -> FAIL ['aspect_ratio']
Last_man_on_earth_1964_formatOK.mp4 -> OK
The_Gun_and_the_Pulpit_formatOK.mp4 -> OK
The_Hill_Gang_Rides_Again_formatOK.mp4 -> OK
Voyage_to_the_Planet_of_Prehistoric_Women_formatOK.mp4 -> OK
